### Implementing baseline models with evasion datasets

This notebook explores the performance of baseline models for spam detection using the **Enron2 dataset** and its **evasion variants** (`charswap`, `homoglyph`, `spacing`). The goal is to evaluate the robustness of machine learning models against adversarial evasion techniques.
#### What we'll do:
1. Train and evaluate baseline models (**Logistic Regression** and **Naive Bayes**) on the original (Enron2) dataset.
2. Assess the performance of these models on evasion datasets when:
   - Models are adversarially trained on evasion datasets.
   - Models are trained only on the original dataset (no adversarial training).
3. Fine-tune the models to optimize their performance.
4. Compare the results to understand the impact of adversarial training.

#### Models Used:
1. **Logistic Regression (LR)**:
   - A linear model that is simple, interpretable, and effective for binary classification tasks like spam detection.
   - It is robust to overfitting when regularization is applied.
2. **Naive Bayes (NB)**:
   - A probabilistic model that assumes feature independence.
   - It is computationally efficient and performs well on text classification tasks, especially when features are sparse (e.g., TF-IDF vectors).

### Importing libraries

In [4]:
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

### Dataset Loading

Loaded the original Enron2 dataset (train, val, test) and its evasion variants (charswap, homoglyph, spacing) from their respective subfolders.
#### Dataset Description:
- **Original Dataset**: Contains spam and ham emails without any modifications.
- **Evasion Datasets**:
  - `charswap`: Emails with character-swapping modifications.
  - `homoglyph`: Emails with homoglyph substitutions (e.g., replacing "o" with "0").
  - `spacing`: Emails with added or removed spaces.

In [5]:
# Define base path to enron2 folder
base_path = "dataset/enron2"

# Load original dataset splits
original_train = pd.read_csv(os.path.join(base_path, "enron2_train.csv"))
original_val = pd.read_csv(os.path.join(base_path, "enron2_val.csv"))
original_test = pd.read_csv(os.path.join(base_path, "enron2_test.csv"))

# Load charswap dataset splits
charswap_train = pd.read_csv(os.path.join(base_path, "charswap/enron2_train_with_charswap.csv"))
charswap_val = pd.read_csv(os.path.join(base_path, "charswap/enron2_val_with_charswap.csv"))
charswap_test = pd.read_csv(os.path.join(base_path, "charswap/enron2_test_with_charswap.csv"))

# Load homoglyph dataset splits
homoglyph_train = pd.read_csv(os.path.join(base_path, "homoglyph/enron2_train_homoglyph.csv"))
homoglyph_val = pd.read_csv(os.path.join(base_path, "homoglyph/enron2_val_homoglyph.csv"))
homoglyph_test = pd.read_csv(os.path.join(base_path, "homoglyph/enron2_test_homoglyph.csv"))

# Load spacing dataset splits
spacing_train = pd.read_csv(os.path.join(base_path, "spacing/enron2_train_spacing.csv"))
spacing_val = pd.read_csv(os.path.join(base_path, "spacing/enron2_val_spacing.csv"))
spacing_test = pd.read_csv(os.path.join(base_path, "spacing/enron2_test_spacing.csv"))

# Check the shape of one dataset to confirm loading
print("Original Train Shape:", original_train.shape)
print(original_train.head())
print("Charswap Train Shape:", charswap_train.shape)
print(charswap_train.head())
print("Homoglyph Train Shape:", homoglyph_train.shape)
print(homoglyph_train.head())
print("Spacing Train Shape:", spacing_train.shape)
print(spacing_train.head())

Original Train Shape: (3727, 2)
                                               email target
0  Subject: membership in the nsf vince : karen m...    ham
1  Subject: holiday gift thank you so much for yo...    ham
2  Subject: re : real options vince , if you take...    ham
3  Subject: men charset = windows - 1252 " > vigo...   spam
4  Subject: vaal medz how t pestilent o save on y...   spam
Charswap Train Shape: (3727, 4)
                                               email target  \
0  Subject: membership in the nsf vince : karen m...    ham   
1  Subject: holiday gift thank you so much for yo...    ham   
2  Subject: re : real options vince , if you take...    ham   
3  Subject: men charset = windows - 1252 " > vigo...   spam   
4  Subject: vaal medz how t pestilent o save on y...   spam   

                                   email_charswapped  was_augmented  
0  Subject: membership in the nsf vince : karen m...          False  
1  Subject: holiday gift thank you so much for yo...     

### Standarization

To ensure consistency across datasets, the following preprocessing steps were applied:
1. **Column Renaming**:
   - For evasion datasets, the modified email content columns (e.g., `email_charswapped`) were renamed to `email_modified`.
   - For the original dataset, the `email` column was renamed to `email_modified`.
2. **Label Encoding**:
   - The `target` column was encoded as:
     - `ham` → 0
     - `spam` → 1

These steps ensure that all datasets have a uniform structure, making them compatible with the machine learning pipeline.

In [6]:
# Function to rename columns and encode labels
def standardize_dataset(df, modified_text_column=None):
    # Create a copy to avoid modifying the original dataframe
    df = df.copy()
    
    # If there's a modified text column, rename it to 'email_modified'
    if modified_text_column:
        df = df.rename(columns={modified_text_column: "email_modified"})
        # Keep only 'email_modified' and 'target' columns, drop others
        df = df[["email_modified", "target"]]
    else:
        # For original dataset, rename 'email' to 'email_modified'
        df = df.rename(columns={"email": "email_modified"})
    
    # Encode the 'target' column: "ham" -> 0, "spam" -> 1
    le = LabelEncoder()
    df["target"] = le.fit_transform(df["target"])
    
    return df

In [7]:
# Standardize all datasets
# Original dataset (no modified text column)
original_train = standardize_dataset(original_train)
original_val = standardize_dataset(original_val)
original_test = standardize_dataset(original_test)

# Charswap dataset
charswap_train = standardize_dataset(charswap_train, "email_charswapped")
charswap_val = standardize_dataset(charswap_val, "email_charswapped")
charswap_test = standardize_dataset(charswap_test, "email_charswapped")

# Homoglyph dataset
homoglyph_train = standardize_dataset(homoglyph_train, "email_homoglyph")
homoglyph_val = standardize_dataset(homoglyph_val, "email_homoglyph")
homoglyph_test = standardize_dataset(homoglyph_test, "email_homoglyph")

# Spacing dataset
spacing_train = standardize_dataset(spacing_train, "email_spaced")
spacing_val = standardize_dataset(spacing_val, "email_spaced")
spacing_test = standardize_dataset(spacing_test, "email_spaced")

In [8]:
# Check the standardized datasets
print("Standardized Original Train Shape:", original_train.shape)
print(original_train.head())
print("Standardized Charswap Train Shape:", charswap_train.shape)
print(charswap_train.head())
print("Standardized Homoglyph Train Shape:", homoglyph_train.shape)
print(homoglyph_train.head())
print("Standardized Spacing Train Shape:", spacing_train.shape)
print(spacing_train.head())

Standardized Original Train Shape: (3727, 2)
                                      email_modified  target
0  Subject: membership in the nsf vince : karen m...       0
1  Subject: holiday gift thank you so much for yo...       0
2  Subject: re : real options vince , if you take...       0
3  Subject: men charset = windows - 1252 " > vigo...       1
4  Subject: vaal medz how t pestilent o save on y...       1
Standardized Charswap Train Shape: (3727, 2)
                                      email_modified  target
0  Subject: membership in the nsf vince : karen m...       0
1  Subject: holiday gift thank you so much for yo...       0
2  Subject: re : real options vince , if you take...       0
3  ['Subject: men charset = windows - 1252 "> vig...       1
4  ['Subject: aval medz how t pestilent o svae on...       1
Standardized Homoglyph Train Shape: (3727, 2)
                                      email_modified  target
0  Subject: membership in the nsf vince : karen m...       0
1  Subject

### Preprocessing and Vectorization

The email content was converted into numerical features using **TF-IDF vectorization**. This technique transforms text data into a sparse matrix of numerical values, where each value represents the importance of a word in a document relative to the entire dataset.

#### Why TF-IDF?
- It captures the importance of words while reducing the impact of commonly occurring words (e.g., "the", "and").
- It is well-suited for text classification tasks, especially when combined with models like Logistic Regression and Naive Bayes.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Function to preprocess and vectorize text data
def preprocess_and_vectorize(train_data, val_data, test_data, text_column="email_modified"):
    vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
    
    # Fit the vectorizer on the training data and transform all splits
    X_train = vectorizer.fit_transform(train_data[text_column])
    X_val = vectorizer.transform(val_data[text_column])
    X_test = vectorizer.transform(test_data[text_column])
    
    # Extract labels
    y_train = train_data["target"]
    y_val = val_data["target"]
    y_test = test_data["target"]
    
    return X_train, X_val, X_test, y_train, y_val, y_test, vectorizer

In [10]:
# Apply preprocessing to each dataset
datasets = {
    "original": (original_train, original_val, original_test),
    "charswap": (charswap_train, charswap_val, charswap_test),
    "homoglyph": (homoglyph_train, homoglyph_val, homoglyph_test),
    "spacing": (spacing_train, spacing_val, spacing_test)
}

# Dictionary to store vectorized data
vectorized_data = {}
for name, (train, val, test) in datasets.items():
    X_train, X_val, X_test, y_train, y_val, y_test, vectorizer = preprocess_and_vectorize(train, val, test)
    vectorized_data[name] = (X_train, X_val, X_test, y_train, y_val, y_test)
    print(f"{name} - X_train shape: {X_train.shape}, X_val shape: {X_val.shape}, X_test shape: {X_test.shape}")

original - X_train shape: (3727, 5000), X_val shape: (932, 5000), X_test shape: (1165, 5000)
charswap - X_train shape: (3727, 5000), X_val shape: (932, 5000), X_test shape: (1165, 5000)
homoglyph - X_train shape: (3727, 5000), X_val shape: (932, 5000), X_test shape: (1165, 5000)
spacing - X_train shape: (3727, 5000), X_val shape: (932, 5000), X_test shape: (1165, 5000)


### Model (Adversarial) Training and Evaluation

The models were trained on the original dataset as well as adversarially on the evasion datasets. This allows us to evaluate their robustness against different types of evasion techniques.

#### Metrics:
The following metrics were calculated for both validation and test sets:
- **Accuracy**: Overall correctness of the model.
- **Precision**: Proportion of correctly identified spam emails out of all predicted spam emails.
- **Recall**: Proportion of correctly identified spam emails out of all actual spam emails.
- **F1-Score**: Harmonic mean of precision and recall, balancing false positives and false negatives.

#### Models:
1. **Logistic Regression**:
   - Trained with regularization to prevent overfitting.
   - Fine-tuned using `GridSearchCV` to optimize the regularization parameter (`C`).
2. **Naive Bayes**:
   - Trained with Laplace smoothing (`alpha` parameter).
   - Fine-tuned using `GridSearchCV` to find the optimal `alpha` value.

#### Results:
The models were evaluated on both the original and evasion datasets to compare their performance under adversarial conditions.

In [11]:
from sklearn.naive_bayes import MultinomialNB
def train_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, dataset_name):
    # Logistic Regression
    lr_model = LogisticRegression(max_iter=1000)
    lr_model.fit(X_train, y_train)
    
    # Evaluate Logistic Regression on validation set
    y_val_pred_lr = lr_model.predict(X_val)
    val_accuracy_lr = accuracy_score(y_val, y_val_pred_lr)
    val_precision_lr = precision_score(y_val, y_val_pred_lr)
    val_recall_lr = recall_score(y_val, y_val_pred_lr)
    val_f1_lr = f1_score(y_val, y_val_pred_lr)
    
    # Evaluate Logistic Regression on test set
    y_test_pred_lr = lr_model.predict(X_test)
    test_accuracy_lr = accuracy_score(y_test, y_test_pred_lr)
    test_precision_lr = precision_score(y_test, y_test_pred_lr)
    test_recall_lr = recall_score(y_test, y_test_pred_lr)
    test_f1_lr = f1_score(y_test, y_test_pred_lr)
    
    # Naive Bayes
    nb_model = MultinomialNB()
    nb_model.fit(X_train, y_train)
    
    # Evaluate Naive Bayes on validation set
    y_val_pred_nb = nb_model.predict(X_val)
    val_accuracy_nb = accuracy_score(y_val, y_val_pred_nb)
    val_precision_nb = precision_score(y_val, y_val_pred_nb)
    val_recall_nb = recall_score(y_val, y_val_pred_nb)
    val_f1_nb = f1_score(y_val, y_val_pred_nb)
    
    # Evaluate Naive Bayes on test set
    y_test_pred_nb = nb_model.predict(X_test)
    test_accuracy_nb = accuracy_score(y_test, y_test_pred_nb)
    test_precision_nb = precision_score(y_test, y_test_pred_nb)
    test_recall_nb = recall_score(y_test, y_test_pred_nb)
    test_f1_nb = f1_score(y_test, y_test_pred_nb)
    
    # Print results
    print(f"\nResults for {dataset_name}:")
    print("Logistic Regression - Validation Set:")
    print(f"Accuracy: {val_accuracy_lr:.4f}, Precision: {val_precision_lr:.4f}, Recall: {val_recall_lr:.4f}, F1: {val_f1_lr:.4f}")
    print("Logistic Regression - Test Set:")
    print(f"Accuracy: {test_accuracy_lr:.4f}, Precision: {test_precision_lr:.4f}, Recall: {test_recall_lr:.4f}, F1: {test_f1_lr:.4f}")
    print("Naive Bayes - Validation Set:")
    print(f"Accuracy: {val_accuracy_nb:.4f}, Precision: {val_precision_nb:.4f}, Recall: {val_recall_nb:.4f}, F1: {val_f1_nb:.4f}")
    print("Naive Bayes - Test Set:")
    print(f"Accuracy: {test_accuracy_nb:.4f}, Precision: {test_precision_nb:.4f}, Recall: {test_recall_nb:.4f}, F1: {test_f1_nb:.4f}")
    
    return lr_model, nb_model

In [12]:
# Update the training loop to store both models
models_lr = {}
models_nb = {}
for name, (X_train, X_val, X_test, y_train, y_val, y_test) in vectorized_data.items():
    lr_model, nb_model = train_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, name)
    models_lr[name] = lr_model
    models_nb[name] = nb_model


Results for original:
Logistic Regression - Validation Set:
Accuracy: 0.9871, Precision: 0.9956, Recall: 0.9540, F1: 0.9744
Logistic Regression - Test Set:
Accuracy: 0.9845, Precision: 0.9930, Recall: 0.9465, F1: 0.9692
Naive Bayes - Validation Set:
Accuracy: 0.9882, Precision: 0.9914, Recall: 0.9623, F1: 0.9766
Naive Bayes - Test Set:
Accuracy: 0.9820, Precision: 0.9929, Recall: 0.9365, F1: 0.9639

Results for charswap:
Logistic Regression - Validation Set:
Accuracy: 0.9903, Precision: 0.9957, Recall: 0.9665, F1: 0.9809
Logistic Regression - Test Set:
Accuracy: 0.9914, Precision: 1.0000, Recall: 0.9666, F1: 0.9830
Naive Bayes - Validation Set:
Accuracy: 0.9925, Precision: 0.9915, Recall: 0.9791, F1: 0.9853
Naive Bayes - Test Set:
Accuracy: 0.9914, Precision: 0.9966, Recall: 0.9699, F1: 0.9831

Results for homoglyph:
Logistic Regression - Validation Set:
Accuracy: 0.9979, Precision: 1.0000, Recall: 0.9916, F1: 0.9958
Logistic Regression - Test Set:
Accuracy: 0.9940, Precision: 1.0000,

In [17]:
from sklearn.model_selection import GridSearchCV
# Fine-tuning Logistic Regression
def fine_tune_LR(X_train, X_val, y_train, y_val, dataset_name):
    param_grid = {"C": [0.01, 0.1, 1, 10, 100]}
    model = LogisticRegression(max_iter=1000)
    grid_search = GridSearchCV(model, param_grid, cv=3, scoring="f1", n_jobs=-1)
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    print(f"\nBest parameters for {dataset_name}: {grid_search.best_params_}")
    y_val_pred = best_model.predict(X_val)
    val_f1 = f1_score(y_val, y_val_pred)
    print(f"Best F1-score on validation set for {dataset_name}: {val_f1:.4f}")
    return best_model

# Fine-tuning Naive Bayes
def fine_tune_naive_bayes(X_train, X_val, y_train, y_val, dataset_name):
    param_grid = {"alpha": [0.01, 0.1, 0.5, 1, 5, 10]}  # Possible values for alpha
    model = MultinomialNB()
    grid_search = GridSearchCV(model, param_grid, cv=3, scoring="f1", n_jobs=-1)
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    print(f"\nBest parameters for Naive Bayes ({dataset_name}): {grid_search.best_params_}")
    y_val_pred = best_model.predict(X_val)
    val_f1 = f1_score(y_val, y_val_pred)
    print(f"Best F1-score on validation set for Naive Bayes ({dataset_name}): {val_f1:.4f}")
    return best_model

In [20]:
# Fine-tune Logistics Regression for each dataset
tuned_LR_models = {}
for name, (X_train, X_val, X_test, y_train, y_val, y_test) in vectorized_data.items():
    tuned_model = fine_tune_LR(X_train, X_val, y_train, y_val, name)
    tuned_LR_models[name] = tuned_model
    y_test_pred = tuned_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    print(f"Tuned Test Set Results for Logistics Regression {name}:")
    print(f"Accuracy: {test_accuracy:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, F1: {test_f1:.4f}")
    
# Fine-tune Naive Bayes for each dataset
tuned_nb_models = {}
for name, (X_train, X_val, X_test, y_train, y_val, y_test) in vectorized_data.items():
    tuned_nb_model = fine_tune_naive_bayes(X_train, X_val, y_train, y_val, name)
    tuned_nb_models[name] = tuned_nb_model
    
    # Evaluate the fine-tuned model on the test set
    y_test_pred = tuned_nb_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    print(f"Tuned Test Set Results for Naive Bayes ({name}):")
    print(f"Accuracy: {test_accuracy:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, F1: {test_f1:.4f}")


Best parameters for original: {'C': 100}
Best F1-score on validation set for original: 0.9895
Tuned Test Set Results for Logistics Regression original:
Accuracy: 0.9948, Precision: 0.9900, Recall: 0.9900, F1: 0.9900

Best parameters for charswap: {'C': 100}
Best F1-score on validation set for charswap: 0.9896
Tuned Test Set Results for Logistics Regression charswap:
Accuracy: 0.9948, Precision: 0.9900, Recall: 0.9900, F1: 0.9900

Best parameters for homoglyph: {'C': 100}
Best F1-score on validation set for homoglyph: 0.9958
Tuned Test Set Results for Logistics Regression homoglyph:
Accuracy: 0.9974, Precision: 0.9966, Recall: 0.9933, F1: 0.9950

Best parameters for spacing: {'C': 100}
Best F1-score on validation set for spacing: 0.9958
Tuned Test Set Results for Logistics Regression spacing:
Accuracy: 0.9966, Precision: 1.0000, Recall: 0.9866, F1: 0.9933

Best parameters for Naive Bayes (original): {'alpha': 0.1}
Best F1-score on validation set for Naive Bayes (original): 0.9832
Tuned

### Working without adversarial training

To simulate a real-world scenario where the models are not adversarially trained, they were trained only on the original dataset and evaluated on the evasion datasets.

This experiment highlights the importance of adversarial training by showing how models trained only on clean data perform poorly on adversarially modified data.

The performance metrics (accuracy, precision, recall, F1-score) were calculated for each evasion dataset. These results were compared with those from adversarially trained models to demonstrate the impact of adversarial training.

In [21]:
# Train models on the original dataset and evaluate on evasion datasets
models_original_train = {}

# Train on original dataset
X_train_original, X_val_original, X_test_original, y_train_original, y_val_original, y_test_original = vectorized_data["original"]

# Train Logistic Regression and Naive Bayes on original training data
lr_model_original = LogisticRegression(max_iter=1000)
lr_model_original.fit(X_train_original, y_train_original)

nb_model_original = MultinomialNB()
nb_model_original.fit(X_train_original, y_train_original)

# Store the models
models_original_train["Logistic Regression"] = lr_model_original
models_original_train["Naive Bayes"] = nb_model_original

# Evaluate on evasion datasets
for evasion_name in ["charswap", "homoglyph", "spacing"]:
    print(f"\nEvaluating models trained on original dataset on {evasion_name} dataset...")
    
    # Get the evasion dataset splits
    X_val_evasion, X_test_evasion, y_val_evasion, y_test_evasion = (
        vectorized_data[evasion_name][1],  # Validation features
        vectorized_data[evasion_name][2],  # Test features
        vectorized_data[evasion_name][4],  # Validation labels
        vectorized_data[evasion_name][5],  # Test labels
    )
    
    # Evaluate Logistic Regression
    y_val_pred_lr = lr_model_original.predict(X_val_evasion)
    y_test_pred_lr = lr_model_original.predict(X_test_evasion)
    print(f"Logistic Regression - Validation Set:")
    print(f"Accuracy: {accuracy_score(y_val_evasion, y_val_pred_lr):.4f}, Precision: {precision_score(y_val_evasion, y_val_pred_lr):.4f}, Recall: {recall_score(y_val_evasion, y_val_pred_lr):.4f}, F1: {f1_score(y_val_evasion, y_val_pred_lr):.4f}")
    print(f"Logistic Regression - Test Set:")
    print(f"Accuracy: {accuracy_score(y_test_evasion, y_test_pred_lr):.4f}, Precision: {precision_score(y_test_evasion, y_test_pred_lr):.4f}, Recall: {recall_score(y_test_evasion, y_test_pred_lr):.4f}, F1: {f1_score(y_test_evasion, y_test_pred_lr):.4f}")
    
    # Evaluate Naive Bayes
    y_val_pred_nb = nb_model_original.predict(X_val_evasion)
    y_test_pred_nb = nb_model_original.predict(X_test_evasion)
    print(f"Naive Bayes - Validation Set:")
    print(f"Accuracy: {accuracy_score(y_val_evasion, y_val_pred_nb):.4f}, Precision: {precision_score(y_val_evasion, y_val_pred_nb):.4f}, Recall: {recall_score(y_val_evasion, y_val_pred_nb):.4f}, F1: {f1_score(y_val_evasion, y_val_pred_nb):.4f}")
    print(f"Naive Bayes - Test Set:")
    print(f"Accuracy: {accuracy_score(y_test_evasion, y_test_pred_nb):.4f}, Precision: {precision_score(y_test_evasion, y_test_pred_nb):.4f}, Recall: {recall_score(y_test_evasion, y_test_pred_nb):.4f}, F1: {f1_score(y_test_evasion, y_test_pred_nb):.4f}")


Evaluating models trained on original dataset on charswap dataset...
Logistic Regression - Validation Set:
Accuracy: 0.7500, Precision: 0.6667, Recall: 0.0502, F1: 0.0934
Logistic Regression - Test Set:
Accuracy: 0.7536, Precision: 0.7000, Recall: 0.0702, F1: 0.1277
Naive Bayes - Validation Set:
Accuracy: 0.6706, Precision: 0.3867, Recall: 0.4854, F1: 0.4304
Naive Bayes - Test Set:
Accuracy: 0.6953, Precision: 0.4222, Recall: 0.5084, F1: 0.4613

Evaluating models trained on original dataset on homoglyph dataset...
Logistic Regression - Validation Set:
Accuracy: 0.7425, Precision: 0.4615, Recall: 0.0251, F1: 0.0476
Logistic Regression - Test Set:
Accuracy: 0.7468, Precision: 0.6111, Recall: 0.0368, F1: 0.0694
Naive Bayes - Validation Set:
Accuracy: 0.5633, Precision: 0.2614, Recall: 0.3849, F1: 0.3113
Naive Bayes - Test Set:
Accuracy: 0.5571, Precision: 0.2471, Recall: 0.3545, F1: 0.2912

Evaluating models trained on original dataset on spacing dataset...
Logistic Regression - Validati